In [8]:
import os
import shutil

In [9]:
path = '/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/4D_MRI_GI_Roberta/4D_MRI_GI_Roberta/Different ways to view the data/View-Slicewise-Per-Subject'
target_path = '/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset'

In [11]:
for folder in os.listdir(path):
    if '_pngs' in folder:
        folder_path = os.path.join(path, folder)
        for file in os.listdir(folder_path):
            if file.endswith('.png'):
                file_path = os.path.join(folder_path, file)
                new_file_path = os.path.join(target_path, file)
                shutil.copy(file_path, new_file_path)
                print(f'Copied: {file_path} to {new_file_path}')

Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/4D_MRI_GI_Roberta/4D_MRI_GI_Roberta/Different ways to view the data/View-Slicewise-Per-Subject/FD_031_pngs/FD-031-slice-43-mask.png to /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-031-slice-43-mask.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/4D_MRI_GI_Roberta/4D_MRI_GI_Roberta/Different ways to view the data/View-Slicewise-Per-Subject/FD_031_pngs/FD-031-slice-16-mask.png to /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-031-slice-16-mask.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/4D_MRI_GI_Roberta/4D_MRI_GI_Roberta/Different ways to view the data/View-Slicewise-Per-Subject/FD_031_pngs/FD-031-slice-26-mask.png to /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-031-slice-2

In [13]:
images = [file for file in os.listdir('full_dataset') if 'image' in file]

In [15]:
# Generate the training and testing sets

In [16]:
import random

In [18]:
train_size = int(len(images) * 0.8)
test_size = len(images) - train_size

In [19]:
train_set = random.sample(images, train_size)
test_set = [img for img in images if img not in train_set]

In [20]:
import pandas as pd

In [21]:
pd.DataFrame(train_set).to_csv('train_set.csv', index=False, header=False)
pd.DataFrame(test_set).to_csv('test_set.csv', index=False, header=False)

In [23]:
# Clone the data into the train_dataset and test_dataset folders
for file in train_set:
    mask_file = file.replace('image', 'mask')
    file_path = os.path.join(target_path, file)
    new_file_path = os.path.join('train_dataset', file)
    shutil.copy(file_path, new_file_path)

    mask_path = os.path.join(target_path, mask_file)
    new_mask_path = os.path.join('train_dataset', mask_file)
    shutil.copy(mask_path, new_mask_path)

    print(f'Copied: {file_path} to {new_file_path}')

for file in test_set:
    mask_file = file.replace('image', 'mask')
    file_path = os.path.join(target_path, file)
    new_file_path = os.path.join('test_dataset', file)
    shutil.copy(file_path, new_file_path)

    mask_path = os.path.join(target_path, mask_file)
    new_mask_path = os.path.join('test_dataset', mask_file)
    shutil.copy(mask_path, new_mask_path)

    print(f'Copied: {file_path} to {new_file_path}')

Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-031-slice-01-image.png to train_dataset/FD-031-slice-01-image.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-030-slice-29-image.png to train_dataset/FD-030-slice-29-image.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-030-slice-41-image.png to train_dataset/FD-030-slice-41-image.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-030-slice-06-image.png to train_dataset/FD-030-slice-06-image.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-029-slice-25-image.png to train_dataset/FD-029-slice-25-image.png
Copied: /home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset/FD-029-slice-08-image.png to train_dataset/FD-029-slice-08-im

In [27]:
import glob
import numpy as np
from tqdm import tqdm
from PIL import Image
def process_to_rgba(input_dir, output_dir):
    """
    Converts pairs of images and masks in an input directory to RGBA format and saves them to an output directory.
    
    Parameters:
        - input_dir (str): Path to the directory containing image and mask pairs in .png format in "Preprocess".
        - output_dir (str): Path to the directory where RGBA images will be saved in "Input".
    """
    os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist
    
    # Retrieve all .png files and sort them to ensure image-mask pairs are processed together
    paths = sorted(glob.glob(os.path.join(input_dir, '*.png')))
    
    # Loop over image and mask pairs, assuming each image is followed by its mask
    for i in tqdm(range(0, len(paths), 2)):
        image_path = paths[i]       # Path to the MRI image
        mask_path = paths[i + 1]    # Path to the corresponding mask
        
        # Load image and mask
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        
        # Check dimensions of the image and the mask match
        if image.size != mask.size:
            raise ValueError(f"Image and mask sizes do not match for {image_path} and {mask_path}: {image.size} vs {mask.size}")
        
        # Convert images to numpy arrays
        image_array = np.array(image)
        mask_array = np.array(mask)
        
        # Stack RGB image and grayscale mask to create an RGBA image
        rgba_image = np.dstack((image_array, mask_array))
        
        # Convert back to PIL RGBA image
        rgba_image_pil = Image.fromarray(rgba_image, 'RGBA')
        
        # Save the RGBA image
        output_filename = os.path.basename(image_path).replace('_image.png', '_RGBA.png')
        output_path = os.path.join(output_dir, output_filename)
        rgba_image_pil.save(output_path)
        
        print(f"Saved 4D image to {output_path}")

In [28]:
process_to_rgba('full_dataset', 'full_dataset_rgba')

  4%|▍         | 14/360 [00:00<00:05, 67.20it/s]

Saved 4D image to full_dataset_rgba/FD-027-slice-01-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-02-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-03-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-04-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-05-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-06-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-07-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-08-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-09-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-10-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-11-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-12-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-13-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-14-image.png


  6%|▌         | 21/360 [00:00<00:05, 64.58it/s]

Saved 4D image to full_dataset_rgba/FD-027-slice-15-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-16-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-17-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-18-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-19-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-20-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-21-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-22-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-23-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-24-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-25-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-26-image.png
Saved 4D image to full_dataset_rgba/FD-027-slice-27-image.png


 10%|▉         | 35/360 [00:00<00:05, 61.63it/s]

...

Saved 4D image to full_dataset_rgba/FD-032-slice-38-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-39-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-40-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-41-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-42-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-43-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-44-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-45-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-46-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-47-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-48-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-49-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-50-image.png


 96%|█████████▌| 345/360 [00:04<00:00, 66.65it/s]

Saved 4D image to full_dataset_rgba/FD-032-slice-51-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-52-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-53-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-54-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-55-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-56-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-57-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-58-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-59-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-60-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-61-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-62-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-63-image.png


100%|██████████| 360/360 [00:05<00:00, 70.51it/s]

Saved 4D image to full_dataset_rgba/FD-032-slice-64-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-65-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-66-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-67-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-68-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-69-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-70-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-71-image.png
Saved 4D image to full_dataset_rgba/FD-032-slice-72-image.png
